# Modelo genérico · Regressão Polinomial e diagnóstico de viés × variância

**Template reutilizável** para avaliar e melhorar um modelo com o protocolo **treino / validação cruzada (CV) / teste**: escolher o grau do polinômio, ajustar a regularização, ler a curva de aprendizado e, numa segunda parte opcional, comparar redes neurais simples, complexas e regularizadas em classificação.

| | |
|---|---|
| **Modelo no sistema** | Machine Learning › 2. Aprendizado supervisionado › cartão *Regressão Polinomial* (`/modelar/aprendizado-supervisionado/#m-polynomial`). Conceitos de apoio: *8. Avaliação e validação*, *9. Seleção de modelos e tuning* e *6. Regularização* |
| **Origem** | `C2_W3_Assignment.ipynb` (*Machine Learning Specialization*, DeepLearning.AI e Stanford Online; *Advice for Applying Machine Learning*), revisado, corrigido e generalizado |
| **Tipo de problema** | Parte A: regressão · Parte B (opcional): classificação multiclasse com rede neural |
| **Dependências** | `numpy`, `pandas`, `matplotlib`, `scikit-learn`; parte B: `keras` 3 |

### Como usar
1. Edite apenas a célula **1. Configuração** (`>>> ADAPTAR`): um dataset para a parte A (alvo numérico) e, se quiser, outro para a parte B (alvo categórico).
2. Com os caminhos em `None`, as duas partes rodam com dados sintéticos equivalentes aos do original (curva quadrática com ruído; seis grupos gaussianos).
3. Para pular a parte B (e não precisar de Keras), use `EXECUTAR_PARTE_REDES = False`.

| conjunto | % típico | uso |
|---|:---:|---|
| treino | 60 | ajustar os parâmetros (w, b) |
| validação cruzada | 20 | escolher grau, λ, arquitetura |
| teste | 20 | medir, **uma única vez**, o desempenho em dados novos |

### O que mudou em relação ao notebook original
- `eval_mse` dividia por `2m` mas se chamava "erro quadrático médio": agora calcula o MSE de fato, `(1/m)·Σ(ŷ − y)²`, como no sistema (e é conferido com `mean_squared_error`). A escolha de grau e de λ não muda, porque o fator ½ não altera o mínimo.
- `eval_cat_err` tinha indentação inconsistente dentro do laço; virou uma expressão vetorizada que aceita formas `(m,)` e `(m, 1)`.
- O modelo regularizado era criado com `name=None`, embora o resultado esperado se chamasse `ComplexRegularized`; e os marcadores `### START CODE HERE ###` fechavam blocos no lugar de `### END CODE HERE ###`. Corrigido.
- O erro de **teste** da rede regularizada era calculado e nunca exibido. Agora o teste é usado uma vez, no fim de cada parte, só para o modelo escolhido na validação.
- `gen_data`, `gen_blobs`, `lin_model`, `tune_m` e todos os gráficos vinham de `assigment_utils.py` (ausente); foram reescritos no notebook e funcionam com **qualquer número de features** (o original só aceitava uma, com gráficos em `%matplotlib widget`).

## 1. Configuração

In [ ]:
# >>> ADAPTAR: única célula que precisa ser editada para usar outros datasets.

# --- parte A: regressão (alvo numérico) ---
DADOS_REGRESSAO = dict(caminho=None,            # None = curva quadrática sintética, como no original
                       coluna_alvo="y", colunas_features=None,
                       separador=None, tem_cabecalho=True, nomes_colunas=None)
GRAU_MAXIMO = 9                                 # testa graus 1 … GRAU_MAXIMO
GRAU_REGULARIZACAO = 10                         # grau alto usado para mostrar o efeito de λ
LAMBDAS = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]
LIMITE_TERMOS = 3000                            # pula graus que gerariam mais termos que isto

# --- parte B: classificação com redes neurais (opcional) ---
EXECUTAR_PARTE_REDES = True
DADOS_CLASSIFICACAO = dict(caminho=None,        # None = seis grupos gaussianos sintéticos
                           coluna_alvo="classe", colunas_features=None,
                           separador=None, tem_cabecalho=True, nomes_colunas=None)
ARQ_COMPLEXA = [120, 40]                        # unidades das camadas ocultas
ARQ_SIMPLES = [6]
LAMBDA_L2 = 0.1                                 # regularização L2 do modelo regularizado
LAMBDAS_L2 = [0.0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.3]
TAXA_APRENDIZADO = 0.01
EPOCAS = 300                                    # o original usava 1000; aumente se houver tempo

# --- divisão e reprodutibilidade ---
FRACAO_CV = 0.2
FRACAO_TESTE = 0.2
SEMENTE = 1

In [ ]:
from math import comb
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

rng = np.random.default_rng(SEMENTE)

In [ ]:
def carregar_tabela(caminho, separador=None, tem_cabecalho=True, nomes_colunas=None, planilha=0):
    """Lê CSV/TXT/TSV, Excel, Parquet ou JSON e devolve um DataFrame."""
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")
    ext = caminho.suffix.lower()
    cabecalho = 0 if tem_cabecalho else None
    if ext in {".csv", ".txt", ".tsv"}:
        sep = separador if separador is not None else ("\t" if ext == ".tsv" else None)
        df = pd.read_csv(caminho, sep=sep, engine="python", header=cabecalho)
    elif ext in {".xlsx", ".xls"}:
        df = pd.read_excel(caminho, sheet_name=planilha, header=cabecalho)
    elif ext == ".parquet":
        df = pd.read_parquet(caminho)
    elif ext == ".json":
        df = pd.read_json(caminho)
    else:
        raise ValueError(f"Formato não suportado: {ext}")
    if nomes_colunas is not None:
        df.columns = nomes_colunas
    elif not tem_cabecalho:
        df.columns = [f"coluna_{i}" for i in range(df.shape[1])]
    return df

def montar_preprocessador(X, padronizar=True, remover_primeira=False):
    """Numéricas: mediana + padronização. Categóricas: moda + one-hot.
    remover_primeira=True descarta uma categoria por variável (evita a colinearidade perfeita
    com o intercepto em modelos lineares). Ajuste SÓ no treino (evita vazamento do teste)."""
    numericas = X.select_dtypes(include="number").columns.tolist()
    categoricas = [c for c in X.columns if c not in numericas]
    passos_num = [SimpleImputer(strategy="median")] + ([StandardScaler()] if padronizar else [])
    transformadores = []
    if numericas:
        transformadores.append(("num", make_pipeline(*passos_num), numericas))
    if categoricas:
        transformadores.append(("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                          drop="first" if remover_primeira else None)), categoricas))
    return ColumnTransformer(transformadores, verbose_feature_names_out=False)


def ler(cfg, sintetico):
    """Lê o dataset descrito em cfg (ou gera o sintético) e devolve X (DataFrame), y e os nomes das features."""
    df = sintetico() if cfg["caminho"] is None else carregar_tabela(
        cfg["caminho"], cfg["separador"], cfg["tem_cabecalho"], cfg["nomes_colunas"])
    alvo = cfg["coluna_alvo"]
    if alvo not in df.columns:
        raise KeyError(f"coluna_alvo={alvo!r} não existe. Colunas: {list(df.columns)}")
    df = df.dropna(subset=[alvo])
    feats = cfg["colunas_features"] or [c for c in df.columns if c != alvo]
    return df[feats], df[alvo].to_numpy(), feats


def dividir_tres(X, y, estratificar=False):
    """Treino / CV / teste em uma divisão por índices (as três partes ficam alinhadas)."""
    idx = np.arange(len(y))
    estr = y if estratificar else None
    i_tr, i_resto = train_test_split(idx, test_size=FRACAO_CV + FRACAO_TESTE, random_state=SEMENTE, stratify=estr)
    estr2 = y[i_resto] if estratificar else None
    i_cv, i_te = train_test_split(i_resto, test_size=FRACAO_TESTE / (FRACAO_CV + FRACAO_TESTE),
                                  random_state=SEMENTE, stratify=estr2)
    return i_tr, i_cv, i_te

# Parte A · Regressão polinomial

## 2. Dados e divisão em três conjuntos

In [ ]:
def regressao_sintetica(m=40, escala=0.7):
    """Equivalente ao gen_data do original: y = x² + ruído proporcional, x em [0, 49]."""
    x = np.linspace(0, 49, m)
    y_ideal = x ** 2 + 100
    return pd.DataFrame({"x": x, "y": y_ideal + escala * y_ideal * (rng.random(m) - 0.5)})


X_df, y, feats = ler(DADOS_REGRESSAO, regressao_sintetica)
y = y.astype(float)
i_tr, i_cv, i_te = dividir_tres(X_df, y)
X_tr, X_cv, X_te = X_df.iloc[i_tr], X_df.iloc[i_cv], X_df.iloc[i_te]
y_tr, y_cv, y_te = y[i_tr], y[i_cv], y[i_te]
print(f"treino {len(y_tr)} | CV {len(y_cv)} | teste {len(y_te)} | features: {feats}")

UMA_FEATURE = X_df.shape[1] == 1 and pd.api.types.is_numeric_dtype(X_df.iloc[:, 0])
if UMA_FEATURE:
    col = feats[0]
    for nome, Xp, yp in [("treino", X_tr, y_tr), ("cv", X_cv, y_cv), ("teste", X_te, y_te)]:
        plt.scatter(Xp[col], yp, label=nome, alpha=0.8)
    plt.xlabel(col); plt.ylabel(DADOS_REGRESSAO["coluna_alvo"]); plt.legend(); plt.title("Treino, CV e teste")
    plt.show()

## 3. Erro para avaliação

$$J_{\text{MSE}} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f(\mathbf{x}^{(i)}) - y^{(i)}\right)^2$$

O curso usava $\frac{1}{2m}$ (metade do MSE, a mesma convenção do custo de treino). O fator não muda qual grau ou λ é o melhor; aqui vale a definição usual, a mesma do sistema.

In [ ]:
def eval_mse(y, yhat):
    """Erro quadrático médio. y, yhat: formas (m,) ou (m, 1)."""
    y, yhat = np.ravel(y), np.ravel(yhat)
    return float(np.mean((yhat - y) ** 2))


assert np.isclose(eval_mse([2.3, 4.1], [2.4, 4.2]), mean_squared_error([2.3, 4.1], [2.4, 4.2]))


def n_termos(n_features, grau):
    return comb(n_features + grau, grau) - 1


def lin_model(grau, lambda_=None, X_ref=X_tr):
    """Pré-processamento → termos polinomiais → padronização → LinearRegression (ou Ridge com alpha = λ)."""
    final = LinearRegression() if lambda_ is None else Ridge(alpha=lambda_)
    return make_pipeline(montar_preprocessador(X_ref), PolynomialFeatures(grau, include_bias=False),
                         StandardScaler(), final)

## 4. Treino × teste: o sinal do sobreajuste
Um polinômio de grau alto ajusta o treino quase perfeitamente e falha em dados novos: **alta variância**. Se o erro de teste fosse usado para escolher o modelo, ele deixaria de medir dados novos; por isso existe o terceiro conjunto, o de validação cruzada.

In [ ]:
n_feat = montar_preprocessador(X_tr).fit_transform(X_tr).shape[1]
grau_alto = GRAU_REGULARIZACAO if n_termos(n_feat, GRAU_REGULARIZACAO) <= LIMITE_TERMOS else 2
modelo = lin_model(grau_alto).fit(X_tr, y_tr)
print(f"grau {grau_alto}: erro treino {eval_mse(y_tr, modelo.predict(X_tr)):,.2f} | erro CV {eval_mse(y_cv, modelo.predict(X_cv)):,.2f}")

## 5. Escolha do grau
O erro de treino cai sempre que o grau sobe. O erro de CV cai enquanto o modelo ganha flexibilidade útil e volta a subir quando ele passa a decorar o treino. O grau escolhido é o de **menor erro de CV**; o grau 1 é a regressão linear simples, a comparação que o cartão pede.

In [ ]:
graus = [g for g in range(1, GRAU_MAXIMO + 1) if n_termos(n_feat, g) <= LIMITE_TERMOS]
if len(graus) < GRAU_MAXIMO:
    print(f"graus acima de {graus[-1]} pulados: gerariam mais de {LIMITE_TERMOS} termos com {n_feat} features")
err_train, err_cv, modelos_grau = [], [], {}
for g in graus:
    m_ = lin_model(g).fit(X_tr, y_tr)
    modelos_grau[g] = m_
    err_train.append(eval_mse(y_tr, m_.predict(X_tr)))
    err_cv.append(eval_mse(y_cv, m_.predict(X_cv)))
grau_otimo = graus[int(np.argmin(err_cv))]
display(pd.DataFrame({"grau": graus, "termos": [n_termos(n_feat, g) for g in graus],
                      "erro treino": err_train, "erro CV": err_cv}).set_index("grau").round(2))
print("grau ótimo pela validação:", grau_otimo)

fig, ax = plt.subplots(1, 2 if UMA_FEATURE else 1, figsize=(12 if UMA_FEATURE else 6, 4), squeeze=False)
if UMA_FEATURE:
    xs = pd.DataFrame({col: np.linspace(X_df[col].min(), X_df[col].max(), 200)})
    ax[0, 0].scatter(X_tr[col], y_tr, s=15, label="treino")
    for g in graus:
        ax[0, 0].plot(xs[col], modelos_grau[g].predict(xs), lw=2.5 if g == grau_otimo else 0.8,
                      label=f"grau {g}" if g in (graus[0], grau_otimo, graus[-1]) else None)
    ax[0, 0].set_ylim(np.min(y) - abs(np.ptp(y)) * 0.2, np.max(y) + abs(np.ptp(y)) * 0.2)
    ax[0, 0].legend(); ax[0, 0].set_title("Ajustes por grau")
a = ax[0, -1]
a.plot(graus, err_train, "o-", label="treino"); a.plot(graus, err_cv, "o-", label="CV")
a.axvline(grau_otimo, ls="--", lw=1); a.set_yscale("log")
a.set_xlabel("grau do polinômio"); a.set_ylabel("MSE (log)"); a.legend(); a.set_title("Viés (esquerda) × variância (direita)")
plt.tight_layout(); plt.show()

## 6. Ajuste da regularização
Mesma lógica com λ (`alpha` do Ridge) num polinômio de grau alto: λ pequeno → variância alta; λ grande → viés alto. O ótimo é o menor erro de CV.

In [ ]:
err_train_l, err_cv_l = [], []
for lam in LAMBDAS:
    m_ = lin_model(grau_alto, lambda_=lam).fit(X_tr, y_tr)
    err_train_l.append(eval_mse(y_tr, m_.predict(X_tr)))
    err_cv_l.append(eval_mse(y_cv, m_.predict(X_cv)))
lambda_otimo = LAMBDAS[int(np.argmin(err_cv_l))]
display(pd.DataFrame({"λ": LAMBDAS, "erro treino": err_train_l, "erro CV": err_cv_l}).set_index("λ").round(2))
print(f"λ ótimo pela validação (grau {grau_alto}):", lambda_otimo)

pos = np.arange(len(LAMBDAS))
plt.plot(pos, err_train_l, "o-", label="treino"); plt.plot(pos, err_cv_l, "o-", label="CV")
plt.axvline(LAMBDAS.index(lambda_otimo), ls="--", lw=1)
plt.xticks(pos, [str(l) for l in LAMBDAS], rotation=45); plt.yscale("log")
plt.xlabel("λ"); plt.ylabel("MSE (log)"); plt.legend(); plt.title("Variância (esquerda) × viés (direita)")
plt.tight_layout(); plt.show()

## 7. Mais dados: curva de aprendizado
Com **variância alta**, aumentar o treino aproxima os erros de treino e de CV. Com **viés alto**, mais dados não ajudam: os dois erros estabilizam altos. A curva abaixo usa o grau alto (propenso a variância) e o grau ótimo.

In [ ]:
fracoes = np.linspace(0.3, 1.0, 8)
curvas = {}
for g in sorted({grau_alto, grau_otimo}):
    et, ec, ms = [], [], []
    for f in fracoes:
        n = max(g + 2, int(round(f * len(y_tr))))
        sub = rng.choice(len(y_tr), size=min(n, len(y_tr)), replace=False)
        m_ = lin_model(g, X_ref=X_tr.iloc[sub]).fit(X_tr.iloc[sub], y_tr[sub])
        et.append(eval_mse(y_tr[sub], m_.predict(X_tr.iloc[sub]))); ec.append(eval_mse(y_cv, m_.predict(X_cv))); ms.append(len(sub))
    curvas[g] = (ms, et, ec)
fig, ax = plt.subplots(1, len(curvas), figsize=(6 * len(curvas), 3.8), squeeze=False)
for a, (g, (ms, et, ec)) in zip(ax[0], curvas.items()):
    a.plot(ms, et, "o-", label="treino"); a.plot(ms, ec, "o-", label="CV"); a.set_yscale("log")
    a.set_xlabel("exemplos de treino (m)"); a.set_ylabel("MSE (log)"); a.set_title(f"Curva de aprendizado, grau {g}"); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
def diagnostico(err_tr, err_cv, base=None):
    """Leitura simples de viés × variância. base = erro de referência (humano, sistema atual), se houver."""
    msgs = []
    if base is not None and err_tr > 1.5 * base:
        msgs.append("viés alto: o erro de treino está muito acima da referência (modelo mais flexível, mais features, menos λ)")
    if err_cv > 1.5 * err_tr:
        msgs.append("variância alta: CV muito pior que o treino (mais dados, mais λ, menos features ou grau menor)")
    return msgs or ["sem sinal forte de viés ou variância"]


i = graus.index(grau_otimo)
print(f"grau {grau_otimo}:", *diagnostico(err_train[i], err_cv[i]), sep="\n  ")

## 8. Teste final (uma única vez)
Só agora o conjunto de teste entra, para o modelo escolhido na validação: o melhor entre o grau ótimo sem regularização e o grau alto com o λ ótimo.

In [ ]:
candidatos = {f"grau {grau_otimo}, sem λ": lin_model(grau_otimo),
              f"grau {grau_alto}, λ = {lambda_otimo}": lin_model(grau_alto, lambda_=lambda_otimo)}
cv_cand = {k: eval_mse(y_cv, m_.fit(X_tr, y_tr).predict(X_cv)) for k, m_ in candidatos.items()}
escolhido = min(cv_cand, key=cv_cand.get)
modelo_final_reg = candidatos[escolhido]
print("escolhido na validação:", escolhido)
print(f"MSE · treino {eval_mse(y_tr, modelo_final_reg.predict(X_tr)):,.2f} | CV {cv_cand[escolhido]:,.2f} | "
      f"TESTE {eval_mse(y_te, modelo_final_reg.predict(X_te)):,.2f} | RMSE teste {np.sqrt(eval_mse(y_te, modelo_final_reg.predict(X_te))):,.2f}")

# Parte B · Redes neurais: complexidade e regularização

## 9. Dados de classificação
O erro de avaliação para classes é a **fração de previsões incorretas**:

$$J_{cv} = \frac{1}{m}\sum_{i=0}^{m-1} \mathbb{1}\left[\hat{y}^{(i)} \neq y^{(i)}\right]$$

Os alvos são índices de classe (sem one-hot).

In [ ]:
def eval_cat_err(y, yhat):
    """Fração de classificações incorretas; aceita formas (m,) e (m, 1)."""
    return float(np.mean(np.ravel(y) != np.ravel(yhat)))


assert np.isclose(eval_cat_err(np.array([1, 2, 0]), np.array([1, 2, 3])), 1 / 3)
assert np.isclose(eval_cat_err(np.array([[1], [2], [0], [3]]), np.array([[1], [2], [1], [3]])), 0.25)

if EXECUTAR_PARTE_REDES:
    import keras
    from keras import layers
    from sklearn.preprocessing import LabelEncoder

    def blobs_sinteticos(n=800, desvio=0.4):
        """Equivalente ao gen_blobs do original: seis grupos gaussianos em 2-D, próximos o bastante para se
        sobreporem (um classificador ideal ainda erra parte dos pontos)."""
        from sklearn.datasets import make_blobs
        centros = np.array([[-1, 0], [1, 0], [0, 1], [0, -1], [-2, 1], [-2, -1]])
        X, y = make_blobs(n_samples=n, centers=centros, cluster_std=desvio, random_state=SEMENTE)
        return pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "classe": y})

    Xc_df, yc_bruto, feats_c = ler(DADOS_CLASSIFICACAO, blobs_sinteticos)
    cod = LabelEncoder()
    yc = cod.fit_transform(yc_bruto)
    K = len(cod.classes_)
    j_tr, j_cv, j_te = dividir_tres(Xc_df, yc, estratificar=True)
    prep_c = montar_preprocessador(Xc_df.iloc[j_tr])
    Xc_tr = prep_c.fit_transform(Xc_df.iloc[j_tr]).astype("float32")
    Xc_cv = prep_c.transform(Xc_df.iloc[j_cv]).astype("float32")
    Xc_te = prep_c.transform(Xc_df.iloc[j_te]).astype("float32")
    yc_tr, yc_cv, yc_te = yc[j_tr], yc[j_cv], yc[j_te]
    print(f"classes: {K} | treino {Xc_tr.shape} | CV {Xc_cv.shape} | teste {Xc_te.shape}")

## 10. Modelo complexo, modelo simples e modelo regularizado
Saída linear com `SparseCategoricalCrossentropy(from_logits=True)` e Adam. No Keras, `L2(λ)` soma λ·Σw² à perda de cada camada (sem o fator 1/2m da fórmula do curso), por isso os valores de λ não são diretamente comparáveis aos da parte A.

In [ ]:
if EXECUTAR_PARTE_REDES:
    def criar_rede(unidades, lambda_l2=0.0, nome=None):
        keras.utils.set_random_seed(1234)
        reg = keras.regularizers.L2(lambda_l2) if lambda_l2 else None
        rede = keras.Sequential(
            [keras.Input(shape=(Xc_tr.shape[1],))]
            + [layers.Dense(u, activation="relu", kernel_regularizer=reg, name=f"L{i + 1}") for i, u in enumerate(unidades)]
            + [layers.Dense(K, activation="linear", name=f"L{len(unidades) + 1}")], name=nome)
        rede.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                     optimizer=keras.optimizers.Adam(TAXA_APRENDIZADO))
        rede.fit(Xc_tr, yc_tr, epochs=EPOCAS, verbose=0)
        return rede

    def prever_classe(rede, X):
        return np.argmax(rede.predict(X, verbose=0), axis=1)   # argmax dos logits = argmax da softmax

    redes = {"complexo": criar_rede(ARQ_COMPLEXA, nome="Complex"),
             "simples": criar_rede(ARQ_SIMPLES, nome="Simple"),
             "regularizado": criar_rede(ARQ_COMPLEXA, LAMBDA_L2, nome="ComplexRegularized")}
    tabela = pd.DataFrame({k: {"erro treino": eval_cat_err(yc_tr, prever_classe(r, Xc_tr)),
                               "erro CV": eval_cat_err(yc_cv, prever_classe(r, Xc_cv)),
                               "parâmetros": r.count_params()} for k, r in redes.items()}).T
    display(tabela)

In [ ]:
if EXECUTAR_PARTE_REDES and Xc_tr.shape[1] == 2:
    g1, g2 = np.meshgrid(np.linspace(Xc_tr[:, 0].min() - 1, Xc_tr[:, 0].max() + 1, 250),
                         np.linspace(Xc_tr[:, 1].min() - 1, Xc_tr[:, 1].max() + 1, 250))
    grade = np.c_[g1.ravel(), g2.ravel()].astype("float32")
    fig, ax = plt.subplots(1, len(redes), figsize=(5 * len(redes), 4.2))
    for a, (nome, r) in zip(ax, redes.items()):
        a.contourf(g1, g2, prever_classe(r, grade).reshape(g1.shape), alpha=0.25, levels=np.arange(K + 1) - 0.5)
        a.scatter(Xc_tr[:, 0], Xc_tr[:, 1], c=yc_tr, s=8, label="treino")
        a.scatter(Xc_cv[:, 0], Xc_cv[:, 1], c=yc_cv, s=20, marker="^", edgecolors="k", linewidths=0.3, label="CV")
        a.set_title(f"{nome}: erro CV {tabela.loc[nome, 'erro CV']:.3f}"); a.legend(loc="upper left", fontsize=7)
    plt.tight_layout(); plt.show()

## 11. Varredura de λ
Com o aumento da regularização, os erros de treino e de CV convergem. Escolha o λ de menor erro de CV.

In [ ]:
if EXECUTAR_PARTE_REDES:
    resultados = []
    for lam in LAMBDAS_L2:
        r = criar_rede(ARQ_COMPLEXA, lam)
        resultados.append({"λ": lam, "erro treino": eval_cat_err(yc_tr, prever_classe(r, Xc_tr)),
                           "erro CV": eval_cat_err(yc_cv, prever_classe(r, Xc_cv)), "rede": r})
    res = pd.DataFrame(resultados).set_index("λ")
    display(res[["erro treino", "erro CV"]].round(4))
    pos = np.arange(len(LAMBDAS_L2))
    plt.plot(pos, res["erro treino"], "o-", label="treino"); plt.plot(pos, res["erro CV"], "o-", label="CV")
    plt.xticks(pos, [str(l) for l in LAMBDAS_L2]); plt.xlabel("λ (L2)"); plt.ylabel("erro de classificação")
    plt.legend(); plt.title("Erro × regularização"); plt.show()

## 12. Teste final da parte B (uma única vez)

In [ ]:
if EXECUTAR_PARTE_REDES:
    melhor_lam = res["erro CV"].idxmin()
    finalistas = {k: r for k, r in redes.items()} | {f"complexo, λ = {melhor_lam}": res.loc[melhor_lam, "rede"]}
    cv_f = {k: eval_cat_err(yc_cv, prever_classe(r, Xc_cv)) for k, r in finalistas.items()}
    vencedor = min(cv_f, key=cv_f.get)
    print("escolhido na validação:", vencedor, f"(erro CV {cv_f[vencedor]:.3f})")
    print(f"erro de TESTE: {eval_cat_err(yc_te, prever_classe(finalistas[vencedor], Xc_te)):.3f}")

## 13. Resumo (segundo o cartão e o original)
- Treino para ajustar parâmetros; **CV** para escolher grau, λ, arquitetura e tamanho de dados; **teste** só no fim.
- Treino ≪ CV → **variância alta**: mais dados, mais λ, modelo menor. Treino alto (acima da referência) → **viés alto**: modelo mais flexível, mais features, menos λ.
- Na regressão polinomial, padronize antes da expansão e combine com Ridge. **Prefira boosting ou SVM-RBF** quando houver muitas features ou uma não linearidade complexa e desconhecida.